In [1]:
import pandas as pd
import numpy as np
# Set option to display all columns
pd.set_option('display.max_columns', None)

In [2]:
data = pd.read_csv('Final_trades_micro_input.csv')

data.head()

,open,high,low,close,volume,volatility,transaction_cost,expected_price,RSI,MACD,MACD_signal,MACD_hist,Stoch_k,Stoch_d,OBV,Upper_BB,Middle_BB,Lower_BB,ATR_1,ADX,+DI,-DI,CCI,5_min_volatility,5_min_volume,5_min_TC,timestamp,forecast_6Hr_open,forecast_6Hr_high,forecast_6Hr_low,forecast_6Hr_close,forecast_6Hr_volume,forecast_6Hr_volatility,forecast_6Hr_transaction_cost,shares,Ticker,Inventory,Trade_Set_ID
0,225.764990,226.184211,225.754742,226.111399,128105.784719,0.000617,0.263699,226.256009,58.328092,0.478232,0.059757,0.416639,528.441215,320.221005,5.050387e+06,226.238045,225.887316,225.495167,0.252764,6.746868,21.432554,14.048062,150.952749,0.000066,710337.073644,0.210592,2772.0,224.514465,226.980010,224.760104,226.677843,307883.835570,0.000426,0.263255,5,AAPL,10,set_1
1,225.695517,226.228422,225.699485,226.142868,134500.951306,0.000561,0.263266,226.392018,59.430143,1.001552,0.190677,0.807093,990.388110,590.718877,5.163082e+06,226.280402,225.897156,225.430272,0.296805,-20.921656,21.707757,12.910271,162.150919,0.000060,727613.351270,0.209859,2772.0,224.444992,227.024221,224.704846,226.709312,323253.628960,0.000426,0.263255,1,AAPL,10,set_1
2,225.626043,226.272633,225.644227,226.174337,141215.368397,0.000520,0.263255,226.528027,60.532194,1.524872,0.321596,1.197548,1452.335005,861.216749,5.275776e+06,226.322760,225.906995,225.365377,0.340846,-48.590180,21.982960,11.772481,173.349088,0.000059,745309.808780,0.209845,2772.0,224.375518,227.068432,224.649589,226.740782,339390.691541,0.000426,0.263255,1,AAPL,10,set_1
3,225.556569,226.316844,225.588969,226.205806,148264.973167,0.000491,0.263255,226.664035,61.634244,2.048193,0.452515,1.588003,1914.281901,1131.714621,5.388471e+06,226.365117,225.916835,225.300481,0.384888,-76.258704,22.258163,10.634690,184.547257,0.000059,763436.665453,0.209846,2772.0,224.306045,227.112643,224.594331,226.772251,356333.325846,0.000426,0.263255,1,AAPL,10,set_1
4,225.487096,226.361055,225.533712,226.237275,155666.498384,0.000470,0.263255,226.800044,62.736295,2.571513,0.583435,1.978457,2376.228796,1402.212493,5.501166e+06,226.407475,225.926674,225.235586,0.428929,-103.927228,22.533366,9.496899,195.745426,0.000058,782004.389116,0.209846,2772.0,224.236571,227.156854,224.539073,226.803720,374121.746493,0.000426,0.263255,1,AAPL,10,set_1


In [3]:
print(data.min())

open                                   9.235514
high                                   9.238027
low                                    9.237492
close                                  9.238324
volume                                   2033.0
volatility                             0.000233
transaction_cost                        0.00139
expected_price                         9.216818
RSI                                   14.714271
MACD                                  -5.406119
MACD_signal                           -5.804963
MACD_hist                             -2.324959
Stoch_k                            -3107.019963
Stoch_d                            -2219.837878
OBV                             -6095513.137171
Upper_BB                               9.271952
Middle_BB                              9.233007
Lower_BB                               9.178315
ATR_1                                 -0.054413
ADX                                 -345.772429
+DI                                    -

In [4]:
print(data.max())

open                                           674.504131
high                                           674.033082
low                                            674.568756
close                                          674.039974
volume                                 71042609164.145126
volatility                                       0.009605
transaction_cost                                 1.508665
expected_price                                 673.061421
RSI                                              77.31204
MACD                                            11.559379
MACD_signal                                      3.480715
MACD_hist                                        9.068131
Stoch_k                                       5132.710876
Stoch_d                                       4754.254915
OBV                                       78113647.473032
Upper_BB                                       672.611421
Middle_BB                                      673.149608
Lower_BB      

In [5]:
# !pip uninstall gymnasium shimmy stable-baselines3 -y
!pip install gym==0.26.0 shimmy>=0.2.1 stable-baselines3 torch

In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from gym import spaces


class UNetTransformerEncoder(nn.Module):
    def __init__(self, in_channels, out_channels, features_dim):
        super(UNetTransformerEncoder, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.features_dim = features_dim

        # Define U-Net layers
        self.encoder1 = self._block(in_channels, 64)
        self.encoder2 = self._block(64, 128)
        self.encoder3 = self._block(128, 256)
        self.encoder4 = self._block(256, 512)
        self.bottleneck = self._block(512, 1024)
        self.decoder4 = self._block(1024 + 512, 512)
        self.decoder3 = self._block(512 + 256, 256)
        self.decoder2 = self._block(256 + 128, 128)
        self.decoder1 = self._block(128 + 64, out_channels)

        self.encoder_layer = nn.TransformerEncoderLayer(d_model=features_dim, nhead=8, dropout=0.1)
        self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=6)

    def _block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        #print(f"Input shape: {x.shape}")
        enc1 = self.encoder1(x)
        #print(f"After encoder1: {enc1.shape}")
        enc2 = self.encoder2(F.max_pool1d(enc1, 2))
        #print(f"After encoder2: {enc2.shape}")
        enc3 = self.encoder3(F.max_pool1d(enc2, 2))
        #print(f"After encoder3: {enc3.shape}")
        enc4 = self.encoder4(F.max_pool1d(enc3, 2))
        #print(f"After encoder4: {enc4.shape}")

        bottleneck = self.bottleneck(F.max_pool1d(enc4, 2))
        #print(f"After bottleneck: {bottleneck.shape}")

        dec4 = self.decoder4(torch.cat((F.interpolate(bottleneck, scale_factor=2), enc4), dim=1))
        #print(f"After decoder4: {dec4.shape}")
        dec3 = self.decoder3(torch.cat((F.interpolate(dec4, scale_factor=2), enc3), dim=1))
        #print(f"After decoder3: {dec3.shape}")
        dec2 = self.decoder2(torch.cat((F.interpolate(dec3, scale_factor=2), enc2), dim=1))
        #print(f"After decoder2: {dec2.shape}")
        dec1 = self.decoder1(torch.cat((F.interpolate(dec2, scale_factor=2), enc1), dim=1))
        #print(f"After decoder1: {dec1.shape}")

        # Transformer encoding
        x = self.transformer_encoder(dec1.permute(2, 0, 1)).permute(1, 2, 0)
        #print(f"After transformer encoder: {x.shape}")

        return x


In [7]:

from gym import spaces

class CustomUNetTransformerModel(BaseFeaturesExtractor):
    def __init__(self, observation_space: spaces.Box, features_dim = 256):
        super(CustomUNetTransformerModel, self).__init__(observation_space, features_dim)
        self.embedding = nn.Linear(observation_space.shape[0], features_dim)  # Adapt the input size if necessary
        self.unet_transformer = UNetTransformerEncoder(1, features_dim,features_dim)  # Pass features_dim correctly

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        x = self.embedding(observations)
        x = x.unsqueeze(1)  # Add channel dimension
        x = self.unet_transformer(x)
        x = x.mean(dim=2)  # Global average pooling
        return x

# import torch as th
# import torch.nn as nn
# from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
# from gym import spaces

# class CustomUNetTransformerModel(BaseFeaturesExtractor):
#     def __init__(self, observation_space: spaces.Box, features_dim=512):
#         super(CustomUNetTransformerModel, self).__init__(observation_space, features_dim)
        
#         # Check if the observation space is flat (1D)
#         input_dim = observation_space.shape[0]
        
#         # Define a simple MLP for 1D observations
#         self.fc = nn.Sequential(
#             nn.Linear(input_dim, 256),
#             nn.ReLU(),
#             nn.Linear(256, features_dim)
#         )
        
#     def forward(self, observations: th.Tensor) -> th.Tensor:
#         # Pass observations through the fully connected layers
#         x = self.fc(observations)
#         return x


In [8]:


# from stable_baselines3.common.policies import ActorCriticPolicy
# import torch as th
# from torch import nn
# from stable_baselines3.common.distributions import CategoricalDistribution, DiagGaussianDistribution

# class CustomActorCriticPolicy(ActorCriticPolicy):
#     def __init__(self, observation_space, action_space, lr_schedule, *args, **kwargs):
#         # Extract custom feature extractor parameters if any
#         features_extractor_class = kwargs.pop('features_extractor_class', CustomUNetTransformerModel)
#         features_extractor_kwargs = kwargs.pop('features_extractor_kwargs', {'features_dim': 512})
        
#         super(CustomActorCriticPolicy, self).__init__(
#             observation_space, 
#             action_space, 
#             lr_schedule, 
#             features_extractor_class=features_extractor_class, 
#             features_extractor_kwargs=features_extractor_kwargs,
#             *args, **kwargs
#         )

#         # Define action distributions for discrete and continuous actions
#         self.action_dist_discrete = CategoricalDistribution(2)  # For order type (0 or 1)
#         self.action_dist_continuous = DiagGaussianDistribution(1)  # For limit price

#         # Define a separate network for the discrete and continuous action outputs
#         self.discrete_action_net = nn.Linear(self.features_dim, 2)  # Output dimension is 2 for the order type
#         self.continuous_action_net = nn.Linear(self.features_dim, 1)  # Output dimension is 1 for the limit price

#         # Initialize log_std for the continuous distribution (learnable parameter)
#         self.log_std = nn.Parameter(th.zeros(1))

#         # Define the value network for estimating the value function
#         self.value_net = nn.Linear(self.features_dim, 1)  # Output dimension is 1 for the value function

#     def forward(self, obs: th.Tensor, deterministic=False):
#         features = self.extract_features(obs)

#         # Get action logits for the discrete action
#         discrete_action_logits = self.discrete_action_net(features)

#         # Create the discrete distribution
#         discrete_dist = self.action_dist_discrete.proba_distribution(discrete_action_logits)

#         # Sample or select the discrete action
#         if deterministic:
#             discrete_action = discrete_dist.mode()
#         else:
#             discrete_action = discrete_dist.sample()

#         # Get mean and log_std for the continuous action
#         continuous_action_mean = self.continuous_action_net(features)
#         continuous_dist = self.action_dist_continuous.proba_distribution(continuous_action_mean, self.log_std)

#         if deterministic:
#             continuous_action = continuous_dist.mode()
#         else:
#             continuous_action = continuous_dist.sample()

#         # Combine the actions into a single action tensor
#         action = th.cat([discrete_action.unsqueeze(-1).float(), continuous_action], dim=-1)

#         # Calculate the log probability of the selected actions
#         log_prob = discrete_dist.log_prob(discrete_action) + continuous_dist.log_prob(continuous_action)

#         # Get the value estimate for the current state
#         value = self.value_net(features)

#         return action, value, log_prob

#     def evaluate_actions(self, obs, actions):
#         features = self.extract_features(obs)

#         # Evaluate the discrete action
#         discrete_action_logits = self.discrete_action_net(features)
#         discrete_dist = self.action_dist_discrete.proba_distribution(discrete_action_logits)
#         discrete_log_prob = discrete_dist.log_prob(actions[:, 0].long())

#         # Evaluate the continuous action
#         continuous_action_mean = self.continuous_action_net(features)
#         continuous_dist = self.action_dist_continuous.proba_distribution(continuous_action_mean, self.log_std)
#         continuous_log_prob = continuous_dist.log_prob(actions[:, 1:])
        
#         # Combine the log_probs
#         log_prob = discrete_log_prob + continuous_log_prob

#         # Entropy (sum of entropies of both distributions)
#         entropy = discrete_dist.entropy() + continuous_dist.entropy()

#         return log_prob, entropy

#     def predict(self, obs, deterministic=False):
#         action, _, _ = self.forward(obs, deterministic=deterministic)
#         return action

#     def _predict(self, observation: th.Tensor, deterministic: bool = False):
#         # Implement the missing _predict method
#         actions, _, _ = self.forward(observation, deterministic)
#         return actions


In [9]:
from stable_baselines3.common.policies import ActorCriticPolicy

class CustomTransformerPolicy(ActorCriticPolicy):
    def __init__(self, observation_space, action_space, lr_schedule, *args, **kwargs):
        features_dim = 256  # Define features_dim here or pass it dynamically
        super(CustomTransformerPolicy, self).__init__(observation_space, action_space, lr_schedule, 
                                                      features_extractor_class=CustomUNetTransformerModel, 
                                                      features_extractor_kwargs={'features_dim': features_dim},
                                                      *args, **kwargs)

# class CustomTransformerPolicy(CustomActorCriticPolicy):  # Use CustomActorCriticPolicy as the base
#     def __init__(self, observation_space, action_space, lr_schedule, *args, **kwargs):
#         super(CustomTransformerPolicy, self).__init__(observation_space, action_space, lr_schedule, *args, **kwargs)


In [31]:
# #import numpy as np
# import numpy as np
# from gym import spaces, Env
# import pandas as pd

# class TradingEnvironmentMicro(Env):
#     metadata = {'render.modes': ['human']}

#     def __init__(self, data, preferred_timeframe=390, initial_inventory=100, max_orders=5):
#         super(TradingEnvironmentMicro, self).__init__()
#         self.data = data
#         self.results = []
#         self.cumulative_reward = 0
#         self.current_step = 0
#         self.prev_mid_pricing = 0
#         self.preferred_timeframe = preferred_timeframe
#         self.initial_inventory = initial_inventory
#         self.remaining_inventory = self.initial_inventory
#         self.max_orders = max_orders
#         self.live_orders = []  # Track active limit orders
#         self.time_diff = 1  # Initialize time_diff
#         self.canceled_orders = []  # Track canceled orders

#         # Define state columns
#         self.state_columns = [
#             'open','high','low','close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
#             'Stoch_k', 'Stoch_d', 'expected_price','OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB', 'ATR_1', 'ADX', 
#             '+DI', '-DI', 'CCI', 'transaction_cost','shares', 'forecast_6Hr_open','forecast_6Hr_close','forecast_6Hr_high',
#             'forecast_6Hr_low', 'forecast_6Hr_volatility', 'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost', 'shares'
#         ]

#         # Define action space as a single Box space
#         self.action_space = spaces.Box(
#             low=np.array([0] * (2 + self.max_orders)),  # 2 for order_type and limit_price_scale, rest for cancel actions
#             high=np.array([1] * (2 + self.max_orders)),
#             dtype=np.float32
#         )

#         # Define observation space
#         self.observation_space = spaces.Box(
#             low=-10000, high=10000, shape=(len(self.state_columns),), dtype=np.float32
#         )

#     def _add_noise_to_action(self, action):
#         # Add noise to the order type and limit price
#         noise_action_0 = np.random.normal(0, 0.02)  # Small noise for order type
#         action[0] += noise_action_0
#         action[0] = np.clip(action[0], self.action_space.low[0], self.action_space.high[0])

#         noise_action_1 = np.random.normal(0, 0.2)  # Noise for limit price
#         action[1] += noise_action_1
#         action[1] = np.clip(action[1], self.action_space.low[1], self.action_space.high[1])

#         return action

#     def step(self, action):
#         """Execute one time step within the environment."""
#         action = self._add_noise_to_action(action)
#         if self.current_step >= len(self.data):
#             done = True
#             return np.zeros(len(self.state_columns)), 0.0, done, {}  # Return default values when done

#         order_type = action[0]  # 0: market order, 1: limit order
#         limit_price_scale = action[1]  # Rescale it to 10% above market price
#         cancel_actions = action[2:2 + self.max_orders]  # Extract the cancel actions

#         # Get the current market price
#         market_price = self.data['close'].iloc[self.current_step]
#         limit_price = market_price + limit_price_scale * (0.1 * market_price)

#         # Debugging print statements
#         print(f"Order Type: {'Market' if order_type < 0.5 else 'Limit'} ({order_type}), Limit Price Scale: {limit_price_scale}")
#         print(f"Calculated Limit Price: {limit_price} (Market Price: {market_price})")

#         # Handle cancel actions
#         new_live_orders = []
#         for i, order in enumerate(self.live_orders):
#             if cancel_actions[i] < 0.5:  # Treat values < 0.5 as "keep", >= 0.5 as "cancel"
#                 new_live_orders.append(order)
#             else:
#                 print(f"Cancelling order: {order}")
#                 self.canceled_orders.append(order)

#         self.live_orders = new_live_orders

#         # If the order type is "Limit", place a new limit order
#         if order_type >= 0.5:
#             print(f"Placing new limit order at price: {limit_price}")
#             self.live_orders.append({'price': limit_price, 'volume': self.initial_inventory, 'time_active': 0})
# #         if self.current_step > 0:
# #             current_trade_set = self.data['Trade_Set_ID'].iloc[self.current_step]
# #             previous_trade_set = self.data['Trade_Set_ID'].iloc[self.current_step - 1]
# #             if current_trade_set != previous_trade_set:
# #                 # Print details of the completed trade set
# #                 self.print_trades()
# #                 # Reset alpha decay and time_diff for a new trade set
# #                 self.total_alpha_decay = 0
# #                 self.time_diff = 1  # Reset time_diff to 1 for the new schedule
# #                 self.prev_mid_pricing = 0
# #             else:
# #                 self.time_diff += 1  # Increment time_diff for each trade within the same schedule
# #         else:
# #             self.total_alpha_decay = 0
# #             self.time_diff = 1  # Initialize time_diff for the first trade
#         # Update the live orders' time active
    
#         for order in self.live_orders:
#             order['time_active'] += 1

#         # Calculate the reward using the new reward function
#         trade_row = self.data.iloc[self.current_step]
#         reward = self._calculate_reward(limit_price, trade_row)

#         # Increment step and check if done
#         self.current_step += 1

#         # Record the trade information
#         trade_info = {
#             'step': self.current_step,
#             'timestamp': self.data['timestamp'].iloc[self.current_step - 1],
#             'order_type': 'Market' if order_type < 0.5 else 'Limit',
#             'volume': volume,
#             'execution_price': market_price if order_type < 0.5 else limit_price,
#             'reward': reward
#         }
#         self.results.append(trade_info)
        

#         done = self.current_step >= len(self.data)
        

#         return self._get_state(), reward, done, {}

#     def _calculate_reward(self, execution_price, trade_row, alpha_decay_rate=0.01):
#         # Constants
#         kappa = 0.1
#         expected_price = self.data['expected_price'].iloc[self.current_step]
#         actual_price = execution_price
#         slippage = expected_price - actual_price
#         transaction_costs = self.data['transaction_cost'].iloc[self.current_step]

#         # Alpha decay component
#         total_alpha_decay = trade_row['shares'] * ((1-alpha_decay_rate) ** self.time_diff)

#         # Calculate mid-pricing as the average of the high and low prices
#         mid_pricing = (trade_row['high'] + trade_row['low']) / 2
#         opp_cost = (mid_pricing - self.prev_mid_pricing) * trade_row['shares']
#         mid_slippage = (mid_pricing - self.data['low'].iloc[self.current_step]) * trade_row['shares']
#         e_price_slippage = (self.data['expected_price'].iloc[self.current_step] - self.data['low'].iloc[self.current_step]) * trade_row['shares']

#         self.prev_mid_pricing = mid_pricing

#         # Penalty for active limit orders
#         active_order_penalty = 0
#         for order in self.live_orders:
#             active_order_penalty += order['volume'] * (np.exp(order['time_active']) - 1)

#         # Combining all the components to calculate the reward
#         penalty = (slippage + transaction_costs + total_alpha_decay + opp_cost + mid_slippage + e_price_slippage + active_order_penalty)

#         # Apply utility theory to adjust the penalty, making the reward more sensitive to larger penalties.
#         reward = -penalty - (2 * kappa * (penalty ** 2))

#         print(f"Reward calculated: {reward}")

#         return reward

#     def print_trades(self):
#         """Print the details of all trades executed in the completed trade set."""
#         if self.results:
#             trades_df = pd.DataFrame(self.results)
#             print('--------------------------------------------------')
#             print(f'Trade Set Completed: {self.data["Trade_Set_ID"].iloc[self.current_step - 1]}')
#             print(trades_df.to_string(index=False))

#             if self.canceled_orders:
#                 print("Canceled Orders:")
#                 canceled_orders_df = pd.DataFrame(self.canceled_orders)
#                 print(canceled_orders_df.to_string(index=False))

#             self.results = []  # Clear results after printing to avoid duplicate printing
#             self.canceled_orders = []  # Clear canceled orders after printing

#     def reset(self):
#         """Reset the environment to the initial state."""
#         self.current_step = 0
#         self.cumulative_reward = 0
#         self.results = []
#         self.live_orders = []  # Clear live orders on reset
#         self.prev_mid_pricing = 0  # Reset previous mid-pricing
#         self.canceled_orders = []  # Clear canceled orders on reset
#         return self._get_state()
    
#     def _get_state(self):
#         """Return the current state of the market based on state columns."""
#         if self.current_step >= len(self.data):
#             return np.zeros(len(self.state_columns))  # Return a zero array or some default value
#         market_conditions = self.data[self.state_columns].iloc[self.current_step].values
#         return market_conditions

#     def render(self, mode='human', close=False):
#         """Render the environment's current state."""
#         print('--------------------------------------------------')
#         print(f'Steps: {self.current_step}')
#         print(f'Cumulative reward: {self.cumulative_reward}')
#         self.print_trades()







# Trading environment for Microtrader Layer
# Input: (timestamp, # of shares, )
from gym import spaces, Env
class TradingEnvironmentMicro(Env):
    metadata = {'render.modes': ['human']}
    def __init__(self, data, preferred_timeframe=390, initial_inventory=100, max_orders=5):
        super(TradingEnvironmentMicro, self).__init__()
        self.data = data
        self.results = []
        self.cumulative_reward = 0
        self.current_step = 0
        self.prev_mid_pricing = 0
        self.preferred_timeframe = preferred_timeframe
        self.initial_inventory = initial_inventory
        self.remaining_inventory = self.initial_inventory
        self.max_orders = max_orders
        self.live_orders = []  # Track active limit orders
        self.time_diff = 1  # Initialize time_diff
        self.canceled_orders = []  # Track canceled orders

        # Define state columns
        self.state_columns = [
            'open','high','low','close', 'volume', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 
            'Stoch_k', '+DI','-DI','Stoch_d', 'expected_price','OBV', 'Upper_BB', 'Middle_BB', 'Lower_BB', 'ATR_1', 'ADX',
            'CCI', 'transaction_cost','shares', 'forecast_6Hr_open','forecast_6Hr_close','forecast_6Hr_high',
            'forecast_6Hr_low', 'forecast_6Hr_volatility', 'forecast_6Hr_volume', 'forecast_6Hr_transaction_cost'
        ]

        # Define flattened action space
        # Define action space as a single Box space
        self.action_space = spaces.Box(
            low=np.array([0] * (2 + self.max_orders)),  # 2 for order_type and limit_price_scale, rest for cancel actions
            high=np.array([1] * (2 + self.max_orders)),
            dtype=np.float32
        )

        # Define observation space
        self.observation_space = spaces.Box(
            low=-10000, high=10000, shape=(len(self.state_columns),), dtype=np.float32
        )
        
    def _add_noise_to_action(self, action):
        # Add noise to the first action (order type)
        noise_action_0 = np.random.normal(0, 0.02)  # Small noise for order type
        action[0] += noise_action_0
        action[0] = np.clip(action[0], self.action_space.low[0], self.action_space.high[0])

        # Add noise to the second action (limit price)
        noise_action_1 = np.random.normal(0, 0.2)  # Noise for limit price
        action[1] += noise_action_1
        action[1] = np.clip(action[1], self.action_space.low[1], self.action_space.high[1])

        return action

#     def step(self, action):
#         """Execute one time step within the environment."""
#         # Check if the environment is done
#         action = self._add_noise_to_action(action)
#         if self.current_step >= len(self.data):
#             done = True
#             return np.zeros(len(self.state_columns)), 0.0, done, {}  # Return default values when done

#         order_type = action[0]  # 0: market order, 1: limit order
#         limit_price_scale = action[1] # resacle it to 10000
#         cancel_actions = action[2:2 + self.max_orders]  # Extract the cancel actions

#         print(f"Order Type: {order_type}, Limit Price_scale: {limit_price_scale}, cancel actions: {cancel_actions}")

#         # Use 'market_price' from the data for market orders
#         volume = self.data['shares'].iloc[self.current_step]
#         market_price = self.data['close'].iloc[self.current_step]
#         limit_price = market_price + limit_price_scale * (0.1 * market_price)

#         print(f"Order Type: {order_type}, Limit Price: {limit_price}")

#         #limit_price = action[1] + (1.05*market_price)
#         execution_price = market_price if order_type < 0.5 else limit_price

#         # Check if we are at the start of a new schedule/trade set
#         if self.current_step > 0:
#             current_trade_set = self.data['Trade_Set_ID'].iloc[self.current_step]
#             previous_trade_set = self.data['Trade_Set_ID'].iloc[self.current_step - 1]
#             if current_trade_set != previous_trade_set:
#                 # Print details of the completed trade set
#                 self.print_trades()
#                 # Reset alpha decay and time_diff for a new trade set
#                 self.total_alpha_decay = 0
#                 self.time_diff = 1  # Reset time_diff to 1 for the new schedule
#                 self.prev_mid_pricing = 0
#             else:
#                 self.time_diff += 1  # Increment time_diff for each trade within the same schedule
#         else:
#             self.total_alpha_decay = 0
#             self.time_diff = 1  # Initialize time_diff for the first trade

#         # Calculate the reward using the new reward function
#         trade_row = self.data.iloc[self.current_step]
#         reward = self._calculate_reward(execution_price, trade_row)

#         # Increment step and check if done
#         self.current_step += 1

#         # Record the trade information
#         trade_info = {
#             'step': self.current_step,
#             'timestamp': self.data['timestamp'].iloc[self.current_step - 1],
#             'order_type': 'Market' if order_type < 0.5 else 'Limit',
#             'volume': volume,
#             'execution_price': execution_price,
#             'reward': reward
#         }
#         self.results.append(trade_info)

#         done = self.current_step >= len(self.data)

#        return self._get_state(), reward, done, {}

    def step(self, action):
        """Execute one time step within the environment."""
        action = self._add_noise_to_action(action)
        if self.current_step >= len(self.data):
            done = True
            return np.zeros(len(self.state_columns)), 0.0, done, {}  # Return default values when done

        order_type = action[0]  # 0: market order, 1: limit order
        limit_price_scale = action[1]  # Rescale it to 10% above market price
        cancel_actions = action[2:2 + len(self.live_orders)]  # Extract the cancel actions, limited to the number of live orders

        # Extract the volume (shares) from the observation or environment data
        volume = self.data['shares'].iloc[self.current_step]

        print(f"Order Type: {order_type:.4f}, Limit Price Scale: {limit_price_scale:.4f}, Cancel Actions: {cancel_actions}")

        # Use 'market_price' from the data for market orders
        market_price = self.data['close'].iloc[self.current_step]
        limit_price = market_price + limit_price_scale * (0.1 * market_price)

        print(f"Order Type: {'Market' if order_type < 0.5 else 'Limit'}, Limit Price: {limit_price:.4f}")

        # Determine the execution price based on order type
        execution_price = market_price if order_type < 0.5 else limit_price

        # Handle cancel actions
        new_live_orders = []
        for i, order in enumerate(self.live_orders):
            if i < len(cancel_actions) and cancel_actions[i] >= 0.5:  # Cancel the order if cancel_action is >= 0.5
                print(f"Cancelling order: {order}")
                self.canceled_orders.append(order)
            else:
                new_live_orders.append(order)

        self.live_orders = new_live_orders

        # If the order type is "Limit", place a new limit order
        if order_type >= 0.5:
            print(f"Placing new limit order at price: {limit_price:.4f}")
            self.live_orders.append({'price': limit_price, 'volume': volume, 'time_active': 0})

        # Update the live orders' time active
        for order in self.live_orders:
            order['time_active'] += 1

        # Check if we are at the start of a new schedule/trade set
        if self.current_step > 0:
            current_trade_set = self.data['Trade_Set_ID'].iloc[self.current_step]
            previous_trade_set = self.data['Trade_Set_ID'].iloc[self.current_step - 1]
            if current_trade_set != previous_trade_set:
                # Print details of the completed trade set
                self.print_trades()
                # Reset alpha decay and time_diff for a new trade set
                self.total_alpha_decay = 0
                self.time_diff = 1  # Reset time_diff to 1 for the new schedule
                self.prev_mid_pricing = 0
            else:
                self.time_diff += 1  # Increment time_diff for each trade within the same schedule
        else:
            self.total_alpha_decay = 0
            self.time_diff = 1  # Initialize time_diff for the first trade

        # Calculate the reward using the new reward function
        trade_row = self.data.iloc[self.current_step]
        reward = self._calculate_reward(execution_price, trade_row)

        # Increment step and check if done
        self.current_step += 1

        # Record the trade information
        trade_info = {
            'step': self.current_step,
            'timestamp': self.data['timestamp'].iloc[self.current_step - 1],
            'order_type': 'Market' if order_type < 0.5 else 'Limit',
            'volume': volume,  # Make sure this matches what you use in the inference loop
            'execution_price': execution_price,
            'reward': reward
        }
        self.results.append(trade_info)

        done = self.current_step >= len(self.data)

        return self._get_state(), reward, done, {}

    def render(self, mode='human', close=False):
        """Render the environment's current state."""
        print('--------------------------------------------------')
        print(f'Steps: {self.current_step}')
        print(f'Cumulative reward: {self.cumulative_reward}')
        self.print_trades()  # Ensure print_trades prints volume correctly




    def _calculate_reward(self, execution_price, trade_row, alpha_decay_rate=0.01):
        # Constants
        kappa = 0.1
        expected_price = self.data['expected_price'].iloc[self.current_step]
        actual_price = execution_price
        slippage = expected_price - actual_price
        transaction_costs = self.data['transaction_cost'].iloc[self.current_step]

        # Alpha decay component
        total_alpha_decay = trade_row['shares'] * ((1-alpha_decay_rate) ** self.time_diff)

        # Calculate mid-pricing as the average of the high and low prices
        mid_pricing = (trade_row['high'] + trade_row['low']) / 2
        opp_cost = (mid_pricing - self.prev_mid_pricing) * trade_row['shares']
        mid_slippage = (mid_pricing - self.data['low'].iloc[self.current_step]) * trade_row['shares']
        e_price_slippage = (self.data['expected_price'].iloc[self.current_step] - self.data['low'].iloc[self.current_step]) * trade_row['shares']

        self.prev_mid_pricing = mid_pricing

        # Penalty for active limit orders (exponentially increases the longer an order is active)
        active_order_penalty = 0
        for order in self.live_orders:
            active_order_penalty += order['volume'] * (np.exp(order['time_active']) - 1)

                # Print each component for debugging
        print(f"Slippage: {slippage}")
        print(f"Transaction Costs: {transaction_costs}")
        print(f"Total Alpha Decay: {total_alpha_decay}")
        print(f"Opportunity Cost: {opp_cost}")
        print(f"Mid Slippage: {mid_slippage}")
        print(f"E-Price Slippage: {e_price_slippage}")
        print(f"Active Order Penalty: {active_order_penalty}")

        # Combining all the components to calculate the reward
        penalty = (
            slippage +
            transaction_costs +
            total_alpha_decay +
            opp_cost +
            mid_slippage +
            e_price_slippage +
            active_order_penalty
        )

        # Apply utility theory to adjust the penalty, making the reward more sensitive to larger penalties.
        reward = -penalty - (2 * kappa * (penalty ** 2))

        print(f"Reward calculated: {reward:.4f}")

        return reward

    
    

    def print_trades(self):
        """Print the details of all trades executed in the completed trade set."""
        if self.results:
            trades_df = pd.DataFrame(self.results)
            print('--------------------------------------------------')
            print(f'Trade Set Completed: {self.data["Trade_Set_ID"].iloc[self.current_step - 1]}')
            print(trades_df.to_string(index=False))
            self.results = []  # Clear results after printing to avoid duplicate printing

    def reset(self):
        """Reset the environment to the initial state."""
        self.current_step = 0
        self.cumulative_reward = 0
        self.results = []
        return self._get_state()
    
    def _get_state(self):
        """Return the current state of the market based on state columns."""
        if self.current_step >= len(self.data):
            return np.zeros(len(self.state_columns))  # Return a zero array or some default value
        market_conditions = self.data[self.state_columns].iloc[self.current_step].values
        return market_conditions



In [32]:
from torch.optim.lr_scheduler import _LRScheduler

class CustomLRScheduler(_LRScheduler):
    def __init__(self, optimizer, step_size, gamma=0.1, last_epoch=-1):
        self.step_size = step_size
        self.gamma = gamma
        super().__init__(optimizer, last_epoch)

    def get_lr(self):
        if (self.last_epoch == 0) or (self.last_epoch % self.step_size != 0):
            return [group['lr'] for group in self.optimizer.param_groups]
        return [group['lr'] * self.gamma for group in self.optimizer.param_groups]


In [33]:
from stable_baselines3.common.callbacks import BaseCallback
import torch

class GradientClippingCallback(BaseCallback):
    def __init__(self, clip_value, verbose=0):
        super(GradientClippingCallback, self).__init__(verbose)
        self.clip_value = clip_value

    def _on_step(self) -> bool:
        # Get the optimizer from the policy
        optimizer = self.model.policy.optimizer
        
        # Apply gradient clipping
        torch.nn.utils.clip_grad_norm_(self.model.policy.parameters(), self.clip_value)
        
        return True


In [34]:
# Training Loop for Microtrader

import torch
from stable_baselines3 import PPO
from torch.optim.lr_scheduler import StepLR
from stable_baselines3.common.callbacks import BaseCallback
import numpy as np


# Example definitions for missing variables
scenario = 'large'  # This could be a string or an object depending on your implementation
timeframe = 390  # Timeframe could be '1m', '1h', '1d', etc.
transaction_size = 1000  # Number of shares or units to trade per transaction


# Placeholder data for now



# Define the best hyperparameters
best_hyperparameters = {'learning_rate': 0.0009931989008886031, 'n_steps': 512, 'batch_size': 128, 
                        'gamma': 0.9916829193042708, 'clip_range': 0.21127653449387027, 'n_epochs': 6, 'ent_coef': 0.1}

# Create the trading environment
env = TradingEnvironmentMicro(data, preferred_timeframe=timeframe, initial_inventory=transaction_size)

# Initialize the environment and model
model = PPO(CustomTransformerPolicy, env, verbose=1, **best_hyperparameters)


# Train the model with a learning rate scheduler
class CustomLRScheduler(BaseCallback):
    def __init__(self, learning_rate, step_size, gamma, verbose=0):
        super(CustomLRScheduler, self).__init__(verbose)
        self.learning_rate = learning_rate
        self.step_size = step_size
        self.gamma = gamma
        self.optimizer = None

    def _on_training_start(self):
        # Initialize optimizer
        self.optimizer = self.model.policy.optimizer

    def _on_step(self):
        if self.optimizer is not None and self.n_calls % self.step_size == 0:
            for param_group in self.optimizer.param_groups:
                param_group['lr'] *= self.gamma
            if self.verbose > 0:
                print(f"Step {self.n_calls}: Updated learning rate to {param_group['lr']}")

lr_scheduler_callback = CustomLRScheduler(learning_rate=0.0015, step_size=100, gamma=0.1)

model.learn(total_timesteps=10000, callback=lr_scheduler_callback)

# Save the model
model.save("trading_agent")


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Order Type: 0.0000, Limit Price Scale: 0.6502, Cancel Actions: []
Order Type: Market, Limit Price: 240.8121
Slippage: 0.1446100767736027
Transaction Costs: 0.263698527810231
Total Alpha Decay: 4.95
Opportunity Cost: 1129.8473834024758
Mid Slippage: 1.0736718364211129
E-Price Slippage: 2.506332741741346
Active Order Penalty: 0
Reward calculated: -260505.3582
Order Type: 0.9978, Limit Price Scale: 0.2399, Cancel Actions: []
Order Type: Limit, Limit Price: 231.5676
Placing new limit order at price: 231.5676
Slippage: -5.175607002594205
Transaction Costs: 0.2632663733773623
Total Alpha Decay: 0.9801
Opportunity Cost: -0.005523317593798538
Mid Slippage: 0.26446873567556395
E-Price Slippage: 0.6925330957913332
Active Order Penalty: 1.718281828459045
Reward calculated: 0.9437
Order Type: 0.3271, Limit Price Scale: 1.0000, Cancel Actions: [0.]
Order Type: Market, Limit Price: 248.7918
Slippage: 0.353

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/numpy/core/_methods.py:235: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/stable_baselines3/common/utils.py:59: RuntimeWarning: invalid value encountered in float_scalars
  return np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y


Order Type: 0.0000, Limit Price Scale: 0.0000, Cancel Actions: []
Order Type: Market, Limit Price: 74.5228
Slippage: -0.0007303555333919576
Transaction Costs: 0.0832935689901944
Total Alpha Decay: 9.70299
Opportunity Cost: -0.1953180952912703
Mid Slippage: -0.1640524337227589
E-Price Slippage: -0.15709477824216833
Active Order Penalty: 0
Reward calculated: -26.4523
Order Type: 0.3091, Limit Price Scale: 0.0000, Cancel Actions: []
Order Type: Market, Limit Price: 74.5049
Slippage: -0.001076306622948664
Transaction Costs: 0.0722931875201453
Total Alpha Decay: 10.566556109999999
Opportunity Cost: -0.2148499048200847
Mid Slippage: -0.24061023638979862
E-Price Slippage: -0.21865859565092194
Active Order Penalty: 0
Reward calculated: -29.8185
Order Type: 0.0111, Limit Price Scale: 0.5354, Cancel Actions: []
Order Type: Market, Limit Price: 78.4751
Slippage: -0.0014222577125053704
Transaction Costs: 0.0731980725258732
Total Alpha Decay: 5.7059402994
Opportunity Cost: -0.11719085717476219
Mid 

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/numpy/core/_methods.py:235: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/numpy/core/_methods.py:246: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/stable_baselines3/common/utils.py:59: RuntimeWarning: invalid value encountered in float_scalars
  return np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y


Order Type: 0.0222, Limit Price Scale: 0.1630, Cancel Actions: []
Order Type: Market, Limit Price: 536.2785
Slippage: -0.05454569028267997
Transaction Costs: 0.2040180053405879
Total Alpha Decay: 0.9509900498999999
Opportunity Cost: 0.06883709133910543
Mid Slippage: -0.06270437351906821
E-Price Slippage: -0.23098752071950912
Active Order Penalty: 0
Reward calculated: -1.0289
Order Type: 0.9659, Limit Price Scale: 0.1055, Cancel Actions: []
Order Type: Limit, Limit Price: 533.0591
Placing new limit order at price: 533.0591
--------------------------------------------------
Trade Set Completed: set_21
 step  timestamp order_type  volume  execution_price        reward
  128     2262.0     Market       5       527.466602 -1.398923e+06
  129     2262.0     Market       2       527.519852 -2.732408e+00
  130     2262.0     Market       1       527.573101 -1.191471e+00
  131     2262.0     Market       1       527.626351 -1.109430e+00
  132     2262.0     Market       1       527.679600 -1.02

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/numpy/core/_methods.py:235: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/numpy/core/_methods.py:246: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(x, axis, dtype, out, keepdims=keepdims, where=where)
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/stable_baselines3/common/utils.py:59: RuntimeWarning: invalid value encountered in float_scalars
  return np.nan if var_y == 0 else 1 - np.var(y_true - y_pred) / var_y


Order Type: 0.0253, Limit Price Scale: 0.3011, Cancel Actions: [0.93091494]
Order Type: Market, Limit Price: 58.6768
Cancelling order: {'price': 57.55509668720928, 'volume': 6, 'time_active': 4}
Slippage: -0.9406538618471458
Transaction Costs: 0.0467957723601638
Total Alpha Decay: 92.178405
Opportunity Cost: 5.037402247652487
Mid Slippage: -0.2909300427360151
E-Price Slippage: -89.40077785688715
Active Order Penalty: 0
Reward calculated: -15.4223
Order Type: 0.0769, Limit Price Scale: 0.9025, Cancel Actions: []
Order Type: Market, Limit Price: 62.1615
Slippage: -0.9874619991740374
Transaction Costs: 0.0467957701070004
Total Alpha Decay: 52.832780549999995
Opportunity Cost: 2.916390774957094
Mid Slippage: -0.22445454602067372
E-Price Slippage: -54.34391267381663
Active Order Penalty: 0
Reward calculated: -0.2517
Order Type: 0.0103, Limit Price Scale: 0.7124, Cancel Actions: []
Order Type: Market, Limit Price: 61.1351
Slippage: -1.034270136500929
Transaction Costs: 0.0467957700984544
Tot

In [35]:
ticker_data = pd.read_csv("test_1.csv")
ticker_data.columns

Index(['Unnamed: 0', 'open', 'high', 'low', 'close', 'volume', 'volatility',
       'transaction_cost', 'expected_price', 'RSI', 'MACD', 'MACD_signal',
       'MACD_hist', 'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB', 'Middle_BB',
       'Lower_BB', 'ATR_1', 'ADX', '#NAME?', '#NAME?.1', 'CCI',
       '5_min_volatility', '5_min_volume', '5_min_TC', 'timestamp',
       'forecast_6Hr_open', 'forecast_6Hr_high', 'forecast_6Hr_low',
       'forecast_6Hr_close', 'forecast_6Hr_volume', 'forecast_6Hr_volatility',
       'forecast_6Hr_transaction_cost', 'shares', 'Ticker', 'Inventory',
       'Trade_Set_ID'],
      dtype='object')

In [36]:
ticker_data = ticker_data.rename(columns={"#NAME?": "+DI", "#NAME?.1": "-DI"})
ticker_data.columns

Index(['Unnamed: 0', 'open', 'high', 'low', 'close', 'volume', 'volatility',
       'transaction_cost', 'expected_price', 'RSI', 'MACD', 'MACD_signal',
       'MACD_hist', 'Stoch_k', 'Stoch_d', 'OBV', 'Upper_BB', 'Middle_BB',
       'Lower_BB', 'ATR_1', 'ADX', '+DI', '-DI', 'CCI', '5_min_volatility',
       '5_min_volume', '5_min_TC', 'timestamp', 'forecast_6Hr_open',
       'forecast_6Hr_high', 'forecast_6Hr_low', 'forecast_6Hr_close',
       'forecast_6Hr_volume', 'forecast_6Hr_volatility',
       'forecast_6Hr_transaction_cost', 'shares', 'Ticker', 'Inventory',
       'Trade_Set_ID'],
      dtype='object')

In [37]:
import pandas as pd
from stable_baselines3 import PPO

# Load your CSV data
#ticker_data = pd.read_csv("test_1.csv")

# Initialize the environment with the loaded data
env = TradingEnvironmentMicro(data=ticker_data, preferred_timeframe=390, initial_inventory=100)

# Load the trained model
model = PPO.load("trading_agent.zip")

# Reset the environment to start inference
obs = env.reset()

# Initialize an empty list to store trade details
trade_details = []

for _ in range(len(ticker_data)):
    # Predict the action to take based on the current observation
    action, _states = model.predict(obs)
    
    # Print the data at the current step
    current_step = env.current_step
    print(env.data.iloc[current_step - 1])
    
    # Step through the environment using the predicted action
    obs, rewards, done, info = env.step(action)
    
    # Extract the order type and limit price from the action
    order_type = "Market" if action[0] < 0.5 else "Limit"
    limit_price_scale = action[1]
    execution_price = obs[3]  # Assuming 'close' price is at index 3

    # Calculate the limit price if the order type is "Limit"
    if order_type == "Limit":
        market_price = obs[3]  # Assuming 'close' price is at index 3
        limit_price = market_price + limit_price_scale * (0.1 * market_price)
        execution_price = limit_price
    
    # Extract the volume (shares) directly from the environment's data
    volume = env.data['shares'].iloc[current_step - 1]  # Using current_step - 1 to match the volume used in the step

    # Handle cancel actions
    cancel_actions = action[2:2 + env.max_orders]
    canceled_orders = [i for i, cancel in enumerate(cancel_actions) if cancel >= 0.5]

    # Append the trade details to the list
    trade_details.append({
        'Order Type': order_type,
        'Execution Price': execution_price,
        'Canceled Orders': canceled_orders,
        'Reward': rewards,
        'Volume': volume
    })
    
    print(f"Trade Executed: {order_type} Order at Price: {execution_price}, Volume: {volume}")
    if canceled_orders:
        print(f"Canceled Orders: {canceled_orders}")

    # If the environment is done, break the loop
    if done:
        break

# Render the final state of the environment
env.render()

# If you want to save the trade details to a file or print them all at once:
# for trade in trade_details:
#     print(trade)


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/gym/spaces/box.py:127: UserWarning: WARN: Box bound precision lowered by casting to float32
  logger.warn(f"Box bound precision lowered by casting to {self.dtype}")
/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/stable_baselines3/common/save_util.py:418: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_glo

Unnamed: 0                                 5
open                              180.527173
high                              181.777594
low                               180.615826
close                             180.672005
volume                                 60081
volatility                          0.001329
transaction_cost                    0.083791
expected_price                    180.570001
RSI                                76.670166
MACD                                2.578191
MACD_signal                          1.49895
MACD_hist                           1.092751
Stoch_k                           136.485655
Stoch_d                          1332.233064
OBV                              3914261.354
Upper_BB                          186.973323
Middle_BB                         181.891495
Lower_BB                          176.443248
ATR_1                               -0.14353
ADX                                173.27977
+DI                                31.730752
-DI       